# 01. 프롬프트 템플릿 만들기

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith('CH02-Prompt')

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [3]:
from langchain_core.prompts import PromptTemplate

template = '{country}의 수도는 어디인가요?'

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [4]:
prompt = prompt.format(country='대한민국')
prompt

'대한민국의 수도는 어디인가요?'

In [6]:
prompt = PromptTemplate.from_template(template)

In [7]:
chain = prompt | llm

chain.invoke('대한민국').content

'대한민국의 수도는 서울특별시입니다.'

In [9]:
template = '{country1}과 {country2}의 수도는 각각 어디인가요?'

prompt = PromptTemplate(
    template=template,
    input_variables=['country1'],
    partial_variables={
        'country2': '미국'
    }
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [10]:
prompt.format(country1='대한민국')

'대한민국과 미국의 수도는 각각 어디인가요?'

In [11]:
prompt_partial = prompt.partial(country2='캐나다')
prompt_partial

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '캐나다'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [12]:
prompt_partial.format(country1='대한민국')

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [13]:
chain = prompt_partial | llm

chain.invoke('대한민국').content

'대한민국의 수도는 서울이고, 캐나다의 수도는 오타와입니다.'

In [14]:
chain.invoke({'country1': '대한민국', 'country2': '호주'}).content

'대한민국의 수도는 서울이며 호주의 수도는 캔버라입니다.'

# 02. 부분 변수 활용하기

In [15]:
from datetime import datetime

datetime.now().strftime('%B %d')

'March 02'

In [16]:
def get_today():
    return datetime.now().strftime('%B %d')

In [17]:
prompt = PromptTemplate(
    template='오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요.',
    input_variables=['n'],
    partial_variables={
        'today': get_today
    }
)

In [18]:
prompt.format(n=3)

'오늘의 날짜는 March 02입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해주세요.'

In [19]:
chain = prompt | llm
print(chain.invoke(3).content)

1. 방시혁 (1972년 3월 2일)
2. 다니엘 크레이그 (1968년 3월 2일)
3. 존 보인 (1960년 3월 2일)


In [20]:
print(chain.invoke({'today': 'Jan 02', 'n':3}).content)

1. Taye Diggs - 1971년 1월 2일
2. Kate Bosworth - 1983년 1월 2일
3. Christy Turlington - 1969년 1월 2일


# 03. YAML 파일로부터 프롬프트 템플릿 로드하기

In [21]:
from langchain_core.prompts import load_prompt
prompt = load_prompt('fruit_color.yaml', encoding='utf-8')
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [22]:
prompt.format(fruit='사과')

'사과의 색깔이 뭐야?'

In [25]:
prompt2 = load_prompt('capital.yaml', encoding='utf-8')
print(prompt2.format(country='대한민국'))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품
  #Answer:



In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model_name='gpt-4o', temperature=0) | StrOutputParser()

answer = chain.stream({'country': '대한민국'})
stream_response(answer)

#Answer:
1. 면적: 서울특별시의 면적은 약 605.21 제곱킬로미터로, 대한민국의 수도이자 가장 큰 도시 중 하나입니다.  
2. 인구: 서울의 인구는 약 950만 명으로, 대한민국에서 가장 인구가 많은 도시입니다.  
3. 역사적 장소: 서울에는 경복궁, 창덕궁, 덕수궁 등 조선시대의 궁궐들이 있으며, 한양도성, 종묘 등 유네스코 세계문화유산으로 지정된 역사적 장소들이 많습니다.  
4. 특산품: 서울은 전통과 현대가 조화를 이루는 도시로, 한복, 한지 공예품, 전통 음식인 김치와 떡 등이 유명합니다. 또한, 현대적인 쇼핑과 문화가 발달하여 다양한 상품과 서비스를 제공합니다.

# 04. ChatPromptTemplate

In [27]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template('{country}의 수도는 어디인가요?')
chat_prompt

ChatPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?'), additional_kwargs={})])

In [28]:
chat_prompt.format(country='대한민국')

'Human: 대한민국의 수도는 어디인가요?'

In [29]:
chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다.'),
        ('human', '반가워요!'),
        ('ai', '안녕하세요! 무엇을 도와드릴까요?'),
        ('human', '{user_input}')
    ]
)

In [30]:
messages = chat_template.format_messages(
    name='테디', user_input='당신의 이름은 무엇입니까?'
)
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 테디입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [31]:
llm = ChatOpenAI()
llm.invoke(messages).content

'제 이름은 테디입니다. 어떻게 도와드릴까요?'

In [32]:
chain = chat_template | llm

chain.invoke({'name': 'Teddy', 'user_input': '당신의 이름은 무엇입니까?'}).content

'제 이름은 Teddy입니다. 저에게 궁금한 점이 있으면 언제든지 물어보세요!'

# 05. MessagesPlaceholder

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.'
        ),
        MessagesPlaceholder(variable_name='conversation'),
        ('human', '지금까지의 대화를 {word_count} 단어로 요약합니다.')
    ]
)
chat_prompt

ChatPromptTemplate(input_variables=['conversation', 'word_count'], input_types={'conversation': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annota

In [34]:
formatted_chat_prompt = chat_prompt.format(
    word_count=5,
    conversation=[
        ('human', '안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.'),
        ('ai', '반가워요! 앞으로 잘 부탁드립니다.')
    ]
)

print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.
AI: 반가워요! 앞으로 잘 부탁드립니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [35]:
llm = ChatOpenAI()

chain = chat_prompt | llm | StrOutputParser()

In [36]:
chain.invoke(
    {
        'word_count': 5,
        'conversation': [
            (
                'human',
                '안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.'
            ),
            ('ai',
             '반가워요! 앞으로 잘 부탁드립니다.')
        ],
        
    }
)

'신입 테디, 반가워요! 함께 일하게 돼서 기쁩니다.'

# 06. 퓨샷 프롬프트

In [37]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

examples = [
    {
        'question': '스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?',
        'answer': '''이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        '''
    }
]

In [38]:
example_prompt = PromptTemplate.from_template(
    'Question:\n{question}\nAnswer:\n{answer}'
)

print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        


In [39]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix='Question:\n{question}\nAnswer:',
    input_variables=['question'],
)

question = 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
final_prompt = prompt.format(question=question)
print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [41]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

answer = llm.stream(final_prompt)
stream_response(answer)

이 질문에 추가 질문이 필요한가요: 예
        추가 질문: Google이 창립된 연도는 몇 년인가요?
        중간 답변: Google은 1998년에 창립되었습니다.
        추가 질문: Bill Gates는 1998년에 몇 살이었나요?
        중간 답변: Bill Gates는 1998년에 43세였습니다.
        최종 답변은: 43세

In [44]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix='Question:\n{question}\nAnswer:',
    input_variables=['question'],
)

chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    {'question': 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'}
)
stream_response(answer)

이 질문에 추가 질문이 필요한가요? 
        추가 질문: Bill Gates가 태어난 연도는 어떻게 되나요?
        중간 답변: Bill Gates는 1955년에 태어났습니다.
        추가 질문: Google이 창립된 연도는 어떻게 되나요?
        중간 답변: Google이 창립된 연도는 1998년입니다.
        최종 답변은: Google이 창립된 1998년에 Bill Gates는 43세이었습니다.

# 07. 예제 선택기

In [48]:
from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma('example_selector', OpenAIEmbeddings())

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    Chroma,
    k=1,
)

In [49]:
selected_examples = example_selector.select_examples({'question': question})

question = 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
print(f'입력에 가장 유사한 예시:\n{question}\n')
for example in selected_examples:
    print(f'question:\n{example["question"]}')
    print(f'answer:\n{example["answer"]}')

입력에 가장 유사한 예시:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?

question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        


In [50]:
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix='Question:\n{question}\nAnswer:',
    input_variables=['question']
)

question = 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
example_selector_prompt = prompt.format(question=question)
print(example_selector_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
        추가 질문: 스티브 잡스는 몇 살에 사망했나요?
        중간 답변: 스티브 잡스는 56세에 사망했습니다.
        추가 질문: 아인슈타인은 몇 살에 사망했나요?
        중간 답변: 아인슈타인은 76세에 사망했습니다.
        최종 답변은: 아인슈타인
        

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [51]:
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix='Question:\n{question}\nAnswer:',
    input_variables=['question'],
)

chain = prompt | llm

answer = chain.stream(
    {'question': 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'}
)
stream_response(answer)

이 질문에 추가 질문이 필요하지 않습니다. 구글이 창립된 1998년에 Bill Gates는 43살이었습니다.

In [52]:
examples = [
    {
        'instruction': '당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요',
        'input': '2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공햇으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.',
        'answer': '''
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
    - 회의는 김수진 팀장의 개회사로 시작됨.
    - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
    - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
    - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
    - 박지민은 디지털 마케팅 전략에 대해 발표.
    - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
    - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
    - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
    - 팀원들 간의 아이디어 공유 및 토론.
    - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
    - 다음 회의 날짜 및 시간 확정.
    - 회의록 정리 및 배포는 박지민 담당.
'''
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다. 이를 통해 고객과의 소통이 더 효과적이 될 것이다.",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, 고객과의 소통을 보다 효과적으로 개선할 수 있을 것으로 기대된다.",
    },
]

In [53]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import (
    SemanticSimilarityExampleSelector,
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma('fewshot_chat', OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ('human', '{instruction}:\n{input}'),
        ('ai', '{answer}')
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, OpenAIEmbeddings(), chroma, k=1
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

In [54]:
question = {
    'instruction': '회의록을 작성해 주세요',
    'input': '2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.'
}

example_selector.select_examples(question)

[{'answer': '\n회의록: XYZ 회사 마케팅 전략 회의\n일시: 2023년 12월 25일\n장소: XYZ 회사 회의실\n참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n1. 개회\n    - 회의는 김수진 팀장의 개회사로 시작됨.\n    - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.\n\n2. 시장 동향 개요 (김수진)\n    - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.\n    - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.\n\n3. 디지털 마케팅 전략 (박지민)\n    - 박지민은 디지털 마케팅 전략에 대해 발표.\n    - 온라인 광고와 SEO 최적화 방안에 중점을 둠.\n\n4. 소셜 미디어 캠페인 (이준호)\n    - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.\n    - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.\n\n5. 종합 논의\n    - 팀원들 간의 아이디어 공유 및 토론.\n    - 각 전략에 대한 예산 및 자원 배분에 대해 논의.\n\n6. 마무리\n    - 다음 회의 날짜 및 시간 확정.\n    - 회의록 정리 및 배포는 박지민 담당.\n',
  'instruction': '당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요',
  'input': '2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공햇으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.'}]

In [55]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            'You are a helpful assistant.',
        ),
        few_shot_prompt,
        ('human', '{instruction}\n{input}')
    ]
)

In [56]:
chain = final_prompt | llm

answer = chain.stream(question)
stream_response(answer)


**회의록: ABC 기술 회사 제품 개발 팀 회의**
**일시:** 2023년 12월 26일
**장소:** ABC 기술 회사 회의실

**참석자:**
- 최현수 (프로젝트 매니저)
- 황지연 (주요 개발자)
- 김태영 (UI/UX 디자이너)

**회의 안건:**
1. 개회
    - 회의는 최현수 매니저의 개회로 시작됨.
    - 주간 진행 상황 회의를 목적으로 함.
    
2. 현황 검토
    - 최현수 매니저가 프로젝트의 현재 진행 상황을 소개.
    - 각 팀원이 개인별 업데이트를 제공.

3. 다가오는 마일스톤 계획
    - 팀은 다가오는 마일스톤에 대한 계획을 수립하기로 함.
    - 목표 설정과 업무 분담에 대한 논의 진행.

4. 업무 업데이트
    - 황지연 개발자가 개발 측면에서의 업데이트를 제공.
    - 김태영 디자이너가 UI/UX 디자인 업데이트를 공유.

5. 다음 주 목표
    - 팀은 다음 주까지의 목표 설정을 완료함.
    - 각자의 할 일과 기한을 명확히하고 업무 일정을 조율함.

6. 마무리
    - 다음 회의 일정 및 안건 확정.
    - 회의록은 최현수 매니저가 작성하여 팀원들에게 배포함.

이상으로 ABC 기술 회사 제품 개발 팀의 주간 진행 상황 회의가 성공적으로 마무리되었습니다.

# 09. 목적에 맞는 예제 선택기

In [57]:
from langchain_teddynote.prompts import CustomExampleSelector

custom_selector = CustomExampleSelector(examples, OpenAIEmbeddings())

custom_selector.select_examples({'instruction': '다음 문장으로 회의록을 작성해 주세요'})

[{'instruction': '당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요',
  'input': '2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공햇으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.',
  'answer': '\n회의록: XYZ 회사 마케팅 전략 회의\n일시: 2023년 12월 25일\n장소: XYZ 회사 회의실\n참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n1. 개회\n    - 회의는 김수진 팀장의 개회사로 시작됨.\n    - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.\n\n2. 시장 동향 개요 (김수진)\n    - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.\n    - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.\n\n3. 디지털 마케팅 전략 (박지민)\n    - 박지민은 디지털 마케팅 전략에 대해 발표.\n    - 온라인 광고와 SEO 최적화 방안에 중점을 둠.\n\n4. 소셜 미디어 캠페인 (이준호)\n    - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.\n    - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.\n\n5. 종합 논의\n    - 팀원들 간의 아이디어 공유 및 토론.\n    - 각 전략에 대한 예산 및 자원 배분에 대해 논의.\n\n6. 마무리\n    - 다음 회의 날짜 및 시간 확정.\n    - 회의록 정리 및 배포는 박지민 담당.\n'}]

In [58]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ('human', '{instruction}:\n{input}'),
        ('ai', '{answer}')
    ]
)

custom_fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=custom_selector,
    example_prompt = example_prompt,
)

custom_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            'You are a helpful assistant.'
        ),
        few_shot_prompt,
        ('human', '{instruction}\n{input}')
    ]
)

In [59]:
chain = custom_prompt | llm

question = {
    'instruction': '회의록을 작성해 주세요',
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

stream_response(chain.stream(question))


회의록: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의
일시: 2023년 12월 26일
장소: ABC 기술 회사 회의실
참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)

1. 개회
    - 회의는 최현수 프로젝트 매니저의 개회사로 시작됨.
    - 회의 목적은 새로운 모바일 애플리케이션 프로젝트의 주간 진행 상황 검토 및 다가오는 마일스톤 계획 수립.

2. 개발 진행 상황 업데이트 (황지연)
    - 황지연은 주요 개발자로서 개발 진행 상황 업데이트를 제공.
    - 기술적 이슈 및 해결책에 대한 보고.

3. UI/UX 업데이트 (김태영)
    - 김태영은 UI/UX 디자이너로서 디자인 업데이트를 제공.
    - 사용성 향상 및 디자인 개선사항에 대한 소개.

4. 다음 주 목표 설정
    - 팀은 다음 주까지의 목표를 설정하기로 합의.
    - 각 역할별 작업 우선순위 및 일정 조정에 대한 논의.

5. 마무리
    - 향후 회의 일정 및 주요 안건 확인.
    - 회의록 작성 및 분배는 김태영에게 할당.

이상으로 ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의 회의록을 마치겠습니다.

# 10. LangChain Hub에서 프롬프트 공유하기

In [2]:
from langchainhub import Client

client = Client()
prompt = client.pull('rlm/rag-prompt')


C:\Users\USER\AppData\Local\Temp\ipykernel_16356\1220612136.py:4: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull('rlm/rag-prompt')
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


In [3]:
print(prompt)

{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}


In [5]:
prompt = Client().pull('rlm/rag-prompt:50442af1')
prompt

C:\Users\USER\AppData\Local\Temp\ipykernel_16356\3549319730.py:1: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = Client().pull('rlm/rag-prompt:50442af1')
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


'{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don\'t know the answer, just say that you don\'t know. Use three sentences maximum and keep the answer concise.\\nQuestion: {question} \\nContext: {context} \\nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}'

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    '주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT: {context}\n\nSUMMARY:'
)
prompt

ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT: {context}\n\nSUMMARY:'), additional_kwargs={})])

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from langsmith import Client as LangSmithClient

client = LangSmithClient()
client.push_prompt('simple-summary-korean', object=prompt)

'https://smith.langchain.com/prompts/simple-summary-korean/de93adb6?organizationId=68918b4e-f45d-4e89-ad71-81c8d118a4c8'